In [1]:
"""
Build a summary table of FAST losses by Building Type (broad category) and SLR scenario.
Output: Excel with two tabs — 'Rounded' (e.g. $1.1B) and 'Raw' (full numbers).
"""

"\nBuild a summary table of FAST losses by Building Type (broad category) and SLR scenario.\nOutput: Excel with two tabs — 'Rounded' (e.g. $1.1B) and 'Raw' (full numbers).\n"

## Imports

In [2]:
import pandas as pd
import re
from pathlib import Path
import getpass

USER = getpass.getuser()

## Config

In [3]:
# Folder of CoastalA FAST output CSVs (one per SLR scenario)
a_dir = Path(rf"C:\Users\{USER}\\Documents\GitHub\FAST\UDF")

out_xlsx = a_dir.parent / "FAST_Loss_By_BuildingType_SLR.xlsx"

# SLR scenarios in desired order (Scenario 1, 2, 3, ...)
SCENARIOS = [12, 36, 52, 77, 96, 108]

# Building Type display order
BUILDING_TYPE_ORDER = ["Commercial", "Gov/Institutional", "Healthcare", "Industrial", "Residential"]

# Occupancy code -> broad damage category
OCC_TO_BROADCAT = {
    "RES1":      "Residential",
    "RES1-1SNB": "Residential", "RES1-1SWB": "Residential",
    "RES1-2SNB": "Residential", "RES1-2SWB": "Residential",
    "RES1-3SNB": "Residential", "RES1-3SWB": "Residential",
    "RES1-SLNB": "Residential", "RES1-SLWB": "Residential",
    "RES2": "Residential",
    "RES3A": "Residential", "RES3B": "Residential", "RES3C": "Residential",
    "RES3D": "Residential", "RES3E": "Residential", "RES3F": "Residential",
    "RES4": "Commercial",
    "RES5": "Residential",
    "RES6": "Residential",
    "COM1": "Commercial", "COM2": "Commercial", "COM3": "Commercial", "COM4": "Commercial",
    "COM5": "Commercial", "COM6": "Healthcare", "COM7": "Healthcare",
    "COM8": "Commercial", "COM9": "Commercial", "COM10": "Commercial",
    "AGR1": "Commercial",
    "IND1": "Industrial", "IND2": "Industrial", "IND3": "Industrial",
    "IND4": "Industrial", "IND5": "Industrial", "IND6": "Industrial",
    "REL1": "Gov/Institutional",
    "GOV1": "Gov/Institutional", "GOV2": "Gov/Institutional",
    "EDU1": "Gov/Institutional", "EDU2": "Gov/Institutional",
}

In [4]:
# ------ STEP 1 — Match each SLR number to its CSV file in the folder ------

# Build a dictionary: {SLR_inches: Path_to_CSV}
# We scan the folder once, skipping _sorted duplicates, and grab the file
# whose name contains one of our SCENARIO numbers.
slr_to_file = {}
for p in sorted(a_dir.glob("*.csv")):
    if "sorted" in p.name.lower():
        continue
    # Extract all numbers from the filename, return the first that matches a scenario
    for n in re.findall(r"\d+", p.name):
        if int(n) in SCENARIOS:
            slr_to_file[int(n)] = p
            break

In [5]:
# ------ STEP 2 — Loop over each scenario, aggregate losses by broad category ------

rows = []   # will hold one dict per output row, later turned into a DataFrame

for i, slr in enumerate(SCENARIOS, start=1):
    f = slr_to_file.get(slr)
    if f is None:
        print(f"Scenario {i} (SLR {slr}\"): file not found — skipping")
        continue

    # Read the FAST output for this scenario
    df = pd.read_csv(f, low_memory=False)

    # Map each row's Occ code to its broad category.
    # Tries the full code first; falls back to the part before "-" for variants like RES1-1SNB.
    broad_cats = []
    for code in df["Occ"]:
        if code in OCC_TO_BROADCAT:
            broad_cats.append(OCC_TO_BROADCAT[code])
        elif isinstance(code, str) and "-" in code:
            base = code.split("-")[0]
            broad_cats.append(OCC_TO_BROADCAT.get(base))
        else:
            broad_cats.append(None)
    df["BroadCat"] = broad_cats

    # Warn about any Occ codes we couldn't classify
    unmapped = df.loc[df["BroadCat"].isna(), "Occ"].unique()
    if len(unmapped):
        print(f"  WARNING (SLR {slr}): unmapped Occ codes — {list(unmapped)}")

    # Sum losses by broad category
    grouped = df.groupby("BroadCat")[["BldgLossUSD", "ContentLossUSD", "InventoryLossUSD"]].sum()
    scenario_label = f"Scenario {i} ({slr}\" SLR)"

# ------ One row per building type ------
    scen_bldg = scen_cont = scen_inv = 0.0
    for bt in BUILDING_TYPE_ORDER:
        bldg    = grouped.loc[bt, "BldgLossUSD"]      if bt in grouped.index else 0.0
        content = grouped.loc[bt, "ContentLossUSD"]   if bt in grouped.index else 0.0
        invent  = grouped.loc[bt, "InventoryLossUSD"] if bt in grouped.index else 0.0
        scen_bldg += bldg
        scen_cont += content
        scen_inv  += invent
        rows.append({
            "Water Level Scenario": scenario_label,
            "Building Type":  bt,
            "Total Loss":     bldg + content + invent,
            "Building Loss":  bldg,
            "Content Loss":   content,
            "Inventory Loss": invent,
        })

    # ------ Total row (runs once per scenario, AFTER the for loop) ------
    rows.append({
        "Water Level Scenario": scenario_label,
        "Building Type":  "Total",
        "Total Loss":     scen_bldg + scen_cont + scen_inv,
        "Building Loss":  scen_bldg,
        "Content Loss":   scen_cont,
        "Inventory Loss": scen_inv,
    })

In [6]:
# ------ STEP 3 — Build the raw DataFrame (full numbers) ------
raw = pd.DataFrame(rows)

# ------ STEP 4 — Build the rounded version ($1.1B / $1.1M / $1.1K format) ------
# Walk each money column and reformat values as readable strings.
# Logic: ≥1B -> "$X.YB", ≥1M -> "$X.YM", ≥1K -> "$X.YK", else "$NN"
rounded = raw.copy()
for col in ["Total Loss", "Building Loss", "Content Loss", "Inventory Loss"]:
    formatted = []
    for x in rounded[col]:
        try:
            v = float(x)
        except (TypeError, ValueError):
            formatted.append(x)
            continue
        if pd.isna(v) or v == 0:
            formatted.append("$0")
            continue
        sign = "-" if v < 0 else ""
        absv = abs(v)
        if absv >= 1e9:
            formatted.append(f"{sign}${absv/1e9:.1f}B")
        elif absv >= 1e6:
            formatted.append(f"{sign}${absv/1e6:.1f}M")
        elif absv >= 1e3:
            formatted.append(f"{sign}${absv/1e3:.1f}K")
        else:
            formatted.append(f"{sign}${absv:.0f}")
    rounded[col] = formatted

# ------ STEP 5 — Write both tabs to one Excel file ------
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    rounded.to_excel(writer, sheet_name="Rounded", index=False)
    raw.to_excel(writer, sheet_name="Raw", index=False)

print(f"\nSaved: {out_xlsx}")
print("\n=== Rounded (first tab) ===")
print(rounded.to_string(index=False))


Saved: C:\Users\MeaganBrown\Documents\GitHub\FAST\FAST_Loss_By_BuildingType_SLR.xlsx

=== Rounded (first tab) ===
 Water Level Scenario     Building Type Total Loss Building Loss Content Loss Inventory Loss
 Scenario 1 (12" SLR)        Commercial      $4.1M       $910.4K        $3.1M         $95.6K
 Scenario 1 (12" SLR) Gov/Institutional    $538.2K        $66.0K      $472.2K             $0
 Scenario 1 (12" SLR)        Healthcare         $0            $0           $0             $0
 Scenario 1 (12" SLR)        Industrial    $403.0K            $0           $0        $403.0K
 Scenario 1 (12" SLR)       Residential    $822.1K       $519.6K      $302.5K             $0
 Scenario 1 (12" SLR)             Total      $5.8M         $1.5M        $3.8M        $498.6K
 Scenario 2 (36" SLR)        Commercial     $17.0M         $4.2M       $12.3M        $418.6K
 Scenario 2 (36" SLR) Gov/Institutional     $10.9M         $1.5M        $9.4M             $0
 Scenario 2 (36" SLR)        Healthcare      $1.